# 03 (Kaggle) — ColPali LoRA Fine-tune

**Where to run this:** [kaggle.com/code](https://www.kaggle.com/code) → New Notebook → Settings → Accelerator = **GPU P100** (free, 30 h/week).

**Why Kaggle and not Colab:** the P100 has 16 GB VRAM, doesn't time-out as aggressively as free Colab, and 30 h/week is enough for two full training runs. Free Colab T4 also works (fp16) but is closer to the memory ceiling.

**Inputs you need attached to the notebook:**
- This repo (clone via the cell below, or attach as a Kaggle Dataset).
- A second Kaggle Dataset uploaded by you that contains `data/processed/reports.parquet`, `data/processed/splits.json`, and the CXR images. (Kaggle notebooks don't have direct internet access to your local machine, so you upload these once.)

**Outputs:** the trained LoRA adapter under `/kaggle/working/models/colpali-cxr-lora/`. Download it back to your laptop and commit to the repo at `models/colpali-cxr-lora/final/`.

In [ ]:
# Sanity check the GPU
!nvidia-smi

In [ ]:
# Clone the repo into /kaggle/working
%cd /kaggle/working
!git clone https://github.com/jo3591/assigment2dsai413 cxr-intel || true
%cd cxr-intel

In [ ]:
# Install pinned dependencies (Kaggle has torch pre-installed)
!pip install -q -r requirements-colab.txt
!pip install -q -e .

In [ ]:
# Hugging Face token (set in Kaggle Add-ons → Secrets → 'HF_TOKEN')
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    from huggingface_hub import login; login(os.environ['HF_TOKEN'])
    print('HF login OK')
except Exception as e:
    print('No HF_TOKEN secret found — ColPali base checkpoint is public so this is OK')

In [ ]:
# Link your uploaded data into the expected paths.
# Replace 'mimic-cxr-processed' with the name of YOUR uploaded Kaggle dataset.
import os, pathlib
DATA_ROOT = '/kaggle/input/mimic-cxr-processed'  # <-- edit this
for sub in ['processed/reports.parquet', 'processed/splits.json']:
    src = pathlib.Path(DATA_ROOT) / sub
    dst = pathlib.Path('data') / sub
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() and src.exists():
        os.symlink(src, dst)
        print(f'linked {dst} -> {src}')
    else:
        print(f'{dst}: exists={dst.exists()}, src exists={src.exists()}')

In [ ]:
# Smoke test: load 50 examples, run 10 steps. Verifies the loop before the full run.
!python scripts/train_colpali_lora.py --epochs 1 --batch-size 1 --limit-train 50

In [ ]:
# Full training run. On P100 ~3.5h for 2 epochs over 2,400 pairs.
# If you have less time, reduce --epochs to 1.
!python scripts/train_colpali_lora.py --epochs 2 --batch-size 2

In [ ]:
# Copy the adapter to /kaggle/working so you can download it as a Kaggle notebook output.
!mkdir -p /kaggle/working/colpali-cxr-lora
!cp -r models/colpali-cxr-lora/final/* /kaggle/working/colpali-cxr-lora/
!ls -la /kaggle/working/colpali-cxr-lora/

## After the run

1. In the Kaggle notebook UI, click **Output** → download `colpali-cxr-lora/` as a zip.
2. Unzip into `models/colpali-cxr-lora/final/` in your local repo.
3. `git add models/colpali-cxr-lora && git commit -m "Add trained ColPali LoRA adapter" && git push`.
4. Back in Colab, build the LoRA index: `python scripts/build_indices.py --backend colpali_lora`.